# notebook_sync_derived_aliases

Adds the tree-derived name aliases (MARRIED_NAME, FATHER_SURNAME) to `genealogy.silver_person_alias` after each GEDCOM ingest. **Insert-only**: it never deletes or updates, so manual aliases and hand edits survive. Run after `07_materialise_gold_tables` (needs `silver_family`, `silver_family_child` and `gold_person_life` current); in the "Ingest and process new GEDCOM" job it follows `Materialise_Gold_Tables`. Idempotent: a second run inserts nothing.

Design rules:
- A derived row is skipped when the table already holds a row for the same person, given name, surname and alias type (case-insensitive), whatever its notes or suffix. A manual row therefore suppresses the derived one.
- Derived rows whose tree basis later disappears (divorce edit, corrected father) are **not** removed; delete them by hand if they cause a wrong match.
- A derived row deleted by hand will come back on the next run. Suppress it with a manual row of the same key instead, or change the tree in Ancestry.
- Rules mirror the logic that notebook_01 Cell 1 used before 2026-10-14; change them here, not there.
- The table is never rebuilt (see `.claude/rules/ocr-and-matching.md`).

After new rows land: re-run the notebook_01 matching cells, `refresh_gold_source_coverage`, notebook 03, then notebook B.

## History
- 2026-10-14: created, replacing the destructive rebuild in notebook_01 Cell 1.

In [ ]:
-- %sql
-- Insert-only: add derived aliases the table does not already hold. Never deletes or updates,
-- so manual rows (and manual edits to derived rows) are untouched. Re-running is a no-op.
-- Match key ignores notes/alias_suffix and compares names case-insensitively, so a manual row
-- for the same person, name and type suppresses the derived one.
INSERT INTO genealogy.silver_person_alias
SELECT d.person_gedcom_id, d.alias_given_name, d.alias_surname, d.alias_type, d.alias_suffix, d.notes
FROM (
SELECT DISTINCT
  sf.wife_gedcom_id AS person_gedcom_id,
  wp.given_name     AS alias_given_name,
  hp.surname        AS alias_surname,
  'MARRIED_NAME'    AS alias_type,
  CAST(NULL AS STRING) AS alias_suffix,
  CONCAT('Married name via ', COALESCE(hp.given_name, '[unknown]'), ' ', hp.surname) AS notes
FROM genealogy.silver_family sf
JOIN genealogy.gold_person_life wp ON sf.wife_gedcom_id = wp.person_gedcom_id
JOIN genealogy.gold_person_life hp ON sf.husband_gedcom_id = hp.person_gedcom_id
WHERE hp.surname IS NOT NULL
  AND sf.relationship_type IS NULL          -- not 'partner'
  AND hp.surname != wp.surname              -- surnames already match: no alias needed
UNION ALL
SELECT DISTINCT
  sfc.child_gedcom_id AS person_gedcom_id,
  cp.given_name       AS alias_given_name,
  fp.surname          AS alias_surname,
  'FATHER_SURNAME'    AS alias_type,
  CAST(NULL AS STRING) AS alias_suffix,
  CONCAT('Father surname via ', fp.given_name, ' ', fp.surname) AS notes
FROM genealogy.silver_family_child sfc
JOIN genealogy.silver_family sf    ON sfc.family_gedcom_id = sf.family_gedcom_id
JOIN genealogy.gold_person_life cp ON sfc.child_gedcom_id  = cp.person_gedcom_id
JOIN genealogy.gold_person_life fp ON sf.husband_gedcom_id = fp.person_gedcom_id
WHERE cp.surname != fp.surname
  AND fp.surname IS NOT NULL
  AND cp.surname IS NOT NULL
  AND sfc.father_relationship_type = 'biological'
) d
LEFT ANTI JOIN genealogy.silver_person_alias e
  ON  e.person_gedcom_id = d.person_gedcom_id
  AND UPPER(TRIM(e.alias_given_name)) <=> UPPER(TRIM(d.alias_given_name))
  AND UPPER(TRIM(e.alias_surname))    <=> UPPER(TRIM(d.alias_surname))
  AND e.alias_type = d.alias_type

## Check: derived aliases still missing (expect 0)

In [ ]:
-- %sql
-- Anomaly check: should return 0 rows. Derived rows still missing after the insert
-- (would indicate the insert key and this check have drifted apart).
SELECT COUNT(*) AS missing_derived_aliases
FROM (
SELECT DISTINCT
  sf.wife_gedcom_id AS person_gedcom_id,
  wp.given_name     AS alias_given_name,
  hp.surname        AS alias_surname,
  'MARRIED_NAME'    AS alias_type,
  CAST(NULL AS STRING) AS alias_suffix,
  CONCAT('Married name via ', COALESCE(hp.given_name, '[unknown]'), ' ', hp.surname) AS notes
FROM genealogy.silver_family sf
JOIN genealogy.gold_person_life wp ON sf.wife_gedcom_id = wp.person_gedcom_id
JOIN genealogy.gold_person_life hp ON sf.husband_gedcom_id = hp.person_gedcom_id
WHERE hp.surname IS NOT NULL
  AND sf.relationship_type IS NULL          -- not 'partner'
  AND hp.surname != wp.surname              -- surnames already match: no alias needed
UNION ALL
SELECT DISTINCT
  sfc.child_gedcom_id AS person_gedcom_id,
  cp.given_name       AS alias_given_name,
  fp.surname          AS alias_surname,
  'FATHER_SURNAME'    AS alias_type,
  CAST(NULL AS STRING) AS alias_suffix,
  CONCAT('Father surname via ', fp.given_name, ' ', fp.surname) AS notes
FROM genealogy.silver_family_child sfc
JOIN genealogy.silver_family sf    ON sfc.family_gedcom_id = sf.family_gedcom_id
JOIN genealogy.gold_person_life cp ON sfc.child_gedcom_id  = cp.person_gedcom_id
JOIN genealogy.gold_person_life fp ON sf.husband_gedcom_id = fp.person_gedcom_id
WHERE cp.surname != fp.surname
  AND fp.surname IS NOT NULL
  AND cp.surname IS NOT NULL
  AND sfc.father_relationship_type = 'biological'
) d
LEFT ANTI JOIN genealogy.silver_person_alias e
  ON  e.person_gedcom_id = d.person_gedcom_id
  AND UPPER(TRIM(e.alias_given_name)) <=> UPPER(TRIM(d.alias_given_name))
  AND UPPER(TRIM(e.alias_surname))    <=> UPPER(TRIM(d.alias_surname))
  AND e.alias_type = d.alias_type